# 01 — MoE Expert Routing Demo (Top-2-of-8, From Scratch in NumPy)

Companion notebook to `01-mixtral-architecture-deep-dive.md`. Implements a simplified, conceptual
version of Mixtral's sparse Mixture-of-Experts routing mechanism from scratch in pure NumPy:

1. A small set of 8 "expert" functions standing in for the 8 feed-forward expert blocks at a single
   Transformer layer — here, simple linear transforms (`y = W_i @ x + b_i`) rather than full
   feed-forward networks, to keep the mechanism traceable end to end.
2. A router/gating function that scores all 8 experts for a given token, selects the **top 2**, and
   combines their weighted outputs — the exact mechanism chapter 01 describes in prose.
3. A demonstration that routing is decided **independently per layer** — the same token can route
   through a different pair of experts at a second layer with its own router.

This is a conceptual illustration of the *mechanism*, not a literal re-implementation of Mixtral's
actual trained weights or real feed-forward expert architecture — no real model, no GPU, no network
calls. Everything runs offline with NumPy only.

## 1. Define 8 experts and a router at one Transformer layer

Each expert is a simple linear transform `y = W_i @ x + b_i`, standing in for a feed-forward expert
block. The router is a single linear layer producing one score per expert (`W_gate @ x`), exactly the
"learned gating function scoring all 8 experts" described in chapter 01.

In [1]:
import numpy as np

rng = np.random.default_rng(seed=13)

D_MODEL = 8          # toy hidden dimension (Mixtral's real d_model is far larger)
NUM_EXPERTS = 8       # matches Mixtral's 8 experts per layer
TOP_K = 2             # matches Mixtral's top-2 routing


class ExpertLayer:
    """One Transformer layer's worth of 8 experts + a router, with top-2 routing."""

    def __init__(self, d_model: int, num_experts: int, top_k: int, rng: np.random.Generator, name: str):
        self.name = name
        self.num_experts = num_experts
        self.top_k = top_k
        # Each expert: a small linear transform standing in for a feed-forward block.
        self.expert_W = [rng.normal(scale=0.5, size=(d_model, d_model)) for _ in range(num_experts)]
        self.expert_b = [rng.normal(scale=0.1, size=(d_model,)) for _ in range(num_experts)]
        # The router: one linear layer producing a raw score per expert.
        self.router_W = rng.normal(scale=0.5, size=(num_experts, d_model))

    def expert_forward(self, expert_idx: int, x: np.ndarray) -> np.ndarray:
        return self.expert_W[expert_idx] @ x + self.expert_b[expert_idx]

    def route(self, x: np.ndarray):
        """Score all experts, softmax, pick top-k, return (selected_idx, selected_weight) pairs."""
        logits = self.router_W @ x                      # one raw score per expert
        exp_logits = np.exp(logits - logits.max())        # numerically stable softmax
        probs = exp_logits / exp_logits.sum()
        top_idx = np.argsort(probs)[::-1][: self.top_k]   # indices of the top-k experts by score
        # Re-normalize the selected experts' weights so they sum to 1 among themselves --
        # this mirrors how Mixtral's router weights the outputs of only the *selected* experts.
        top_weights = probs[top_idx]
        top_weights = top_weights / top_weights.sum()
        return list(zip(top_idx.tolist(), top_weights.tolist())), probs

    def forward(self, x: np.ndarray):
        """Combine the top-k selected experts' outputs, weighted by the router's (renormalized) scores."""
        selection, probs = self.route(x)
        output = np.zeros_like(x)
        for expert_idx, weight in selection:
            output += weight * self.expert_forward(expert_idx, x)
        return output, selection, probs


layer1 = ExpertLayer(D_MODEL, NUM_EXPERTS, TOP_K, rng, name="layer_1")
print(f"Built an ExpertLayer with {NUM_EXPERTS} experts, top-{TOP_K} routing, d_model={D_MODEL}.")


Built an ExpertLayer with 8 experts, top-2 routing, d_model=8.


## 2. Route a batch of tokens through the layer

Each "token" here is just a random vector in R^8, standing in for a hidden-state vector at this layer.
For each token we print which 2 experts the router selected and their (renormalized) combination
weights -- this is the from-scratch, traceable version of the router mechanism chapter 01 describes.

In [2]:
token_names = ["tok_A", "tok_B", "tok_C", "tok_D", "tok_E"]
tokens = [rng.normal(size=(D_MODEL,)) for _ in token_names]

print(f"{'token':8s} {'selected experts (idx: weight)':45s} outputs")
print("-" * 90)
layer1_outputs = {}
layer1_selections = {}
for name, x in zip(token_names, tokens):
    out, selection, probs = layer1.forward(x)
    layer1_outputs[name] = out
    layer1_selections[name] = selection
    sel_str = ", ".join(f"{idx}: {w:.3f}" for idx, w in selection)
    print(f"{name:8s} {sel_str:45s} {np.round(out, 3)}")

print()
print("Confirmed: each token independently selects exactly 2 of the 8 experts, weighted by the")
print("router's own (renormalized) confidence in each -- the top-2-of-8 mechanism from chapter 01.")

assert all(len(sel) == TOP_K for sel in layer1_selections.values()), "Every token must route to exactly top_k experts."


token    selected experts (idx: weight)                outputs
------------------------------------------------------------------------------------------
tok_A    7: 0.504, 5: 0.496                            [ 0.06  -0.817 -0.581 -0.173  0.277 -1.229  1.339 -1.161]
tok_B    1: 0.634, 3: 0.366                            [ 1.297  0.118 -1.288 -0.632 -0.969 -0.306  0.258 -0.13 ]
tok_C    7: 0.520, 0: 0.480                            [ 1.014  0.224 -0.572 -1.52  -0.388  1.655 -0.855 -1.526]
tok_D    1: 0.586, 6: 0.414                            [ 0.104  0.473  0.268  0.493 -1.975  1.166  1.962  1.337]
tok_E    2: 0.573, 5: 0.427                            [-0.374 -0.175  0.828 -0.988 -0.831  0.677  0.195 -0.157]

Confirmed: each token independently selects exactly 2 of the 8 experts, weighted by the
router's own (renormalized) confidence in each -- the top-2-of-8 mechanism from chapter 01.


## 3. Prove routing is decided independently per layer

Mixtral's routing decision is made fresh at *every* layer -- there's no single "assigned expert" for a
token across the whole model. We build a second layer with its own independently-initialized router and
route the same tokens through it, then check whether any token happens to get the same expert pair at
both layers (it's fine if one does by chance with only 8 experts -- the point is the decision genuinely
comes from an independent router, not that the two layers must always disagree).

In [3]:
layer2 = ExpertLayer(D_MODEL, NUM_EXPERTS, TOP_K, rng, name="layer_2")

print(f"{'token':8s} {'layer_1 experts':20s} {'layer_2 experts':20s} same pair?")
print("-" * 70)
same_pair_count = 0
for name, x in zip(token_names, tokens):
    _, sel1, _ = layer1.forward(x)
    _, sel2, _ = layer2.forward(x)
    idx1 = tuple(sorted(idx for idx, _ in sel1))
    idx2 = tuple(sorted(idx for idx, _ in sel2))
    same = idx1 == idx2
    same_pair_count += int(same)
    print(f"{name:8s} {str(idx1):20s} {str(idx2):20s} {same}")

print()
print(f"{same_pair_count}/{len(token_names)} tokens happened to select the same expert pair at both layers.")
print("Each layer's router is independent -- routing is a fresh, local decision every layer, exactly")
print("as chapter 01 describes: there is no single 'assigned expert' for a token across the whole model.")


token    layer_1 experts      layer_2 experts      same pair?
----------------------------------------------------------------------
tok_A    (5, 7)               (1, 3)               False
tok_B    (1, 3)               (1, 6)               False
tok_C    (0, 7)               (0, 5)               False
tok_D    (1, 6)               (0, 7)               False
tok_E    (2, 5)               (1, 6)               False

0/5 tokens happened to select the same expert pair at both layers.
Each layer's router is independent -- routing is a fresh, local decision every layer, exactly
as chapter 01 describes: there is no single 'assigned expert' for a token across the whole model.


## Summary

| Section | Demonstrates | Matches |
|---|---|---|
| 1 | 8 experts + a router built from scratch in NumPy | Chapter 01's description of one Transformer layer's MoE block |
| 2 | Top-2 selection and weighted combination for a batch of tokens | Chapter 01's router mechanism: score all 8, select top 2, weight-combine |
| 3 | Two independently-initialized layers routing the same tokens differently | Chapter 01's point that routing is decided fresh, per layer, with no fixed per-token expert assignment |

This is a deliberately simplified, conceptual mechanism demonstration -- real Mixtral experts are full
feed-forward networks operating on much higher-dimensional hidden states, trained end to end via
gradient descent jointly with the router, not random linear transforms. The *routing logic* itself --
score all experts, select the top 2, renormalize and weight-combine their outputs, independently per
layer -- is implemented faithfully to chapter 01's description.